# Test your NL2SH model

Use this notebook to send prompts to your existing Kaggle + ngrok endpoint and
read the model's replies. **No GPU, model download, or package installation is
needed in this testing notebook.** It uses Python's standard library.

1. Keep the **original serving notebook running** in its own session.
2. Open this notebook in local Jupyter, Colab, or a separate Kaggle CPU session
   with **Internet enabled**.
3. Check `ENDPOINT_URL` below. Update it if your serving notebook prints a new URL.
4. Use the **same `NL2SH_API_KEY`** as the serving notebook. Set it as a Kaggle
   Secret and enable notebook access, set a local environment variable, or enter
   it at the hidden prompt. The ngrok authtoken is not needed here.
5. Run the setup and health cells, then edit `PROMPT` and rerun its cell to try
   your own requests. A final cell contains a few sample prompts.

Replies are displayed as text. This notebook does not execute generated shell
commands. Saved cell outputs are initially empty and contain no credentials.


In [ ]:
import getpass
import json
import os
import time
import urllib.error
import urllib.request

ENDPOINT_URL = "https://YOUR-NGROK-DOMAIN.ngrok-free.dev"
# A base URL ending in /v1 or the full chat URL can also be pasted above.
ENDPOINT_URL = ENDPOINT_URL.strip().rstrip("/").removesuffix("/v1/chat/completions").removesuffix("/v1")
if not ENDPOINT_URL.startswith("https://"):
    raise ValueError("Use the HTTPS ngrok endpoint printed by your serving notebook.")

API_KEY = os.environ.get("NL2SH_API_KEY", "").strip()
if not API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        API_KEY = UserSecretsClient().get_secret("NL2SH_API_KEY").strip()
    except Exception:
        API_KEY = getpass.getpass("Enter the serving notebook's NL2SH_API_KEY (hidden): ").strip()
if not API_KEY:
    raise ValueError("NL2SH_API_KEY is required.")

print("Endpoint:", ENDPOINT_URL)
print("API key loaded; value hidden.")


In [ ]:
def send_request(path, payload=None):
    request = urllib.request.Request(
        ENDPOINT_URL + path,
        data=json.dumps(payload).encode() if payload is not None else None,
        headers={
            "Authorization": "Bearer " + API_KEY,
            "Content-Type": "application/json",
            "ngrok-skip-browser-warning": "1",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=180) as response:
            return json.load(response)
    except urllib.error.HTTPError as error:
        detail = error.read(2000).decode(errors="replace").replace(API_KEY, "[REDACTED]")
        hint = {
            401: "Check that NL2SH_API_KEY matches the serving notebook.",
            429: "The GPU is busy. Wait for the current request to finish and retry.",
            404: "Check the ngrok URL and that the serving notebook is still running.",
            502: "Check that the API server and ngrok tunnel are still running.",
            503: "The endpoint is unavailable. Check the serving notebook.",
        }.get(error.code, "Check the response and serving notebook's logs.")
        raise RuntimeError(f"HTTP {error.code}: {hint}\n{detail}") from None
    except urllib.error.URLError as error:
        raise RuntimeError(f"Cannot reach the endpoint: {error.reason}. Enable Internet and check the URL.") from None
    except TimeoutError:
        raise RuntimeError("Request timed out after 180 seconds. Check the serving notebook and retry.") from None
    except json.JSONDecodeError:
        raise RuntimeError("The endpoint returned non-JSON content. Check the ngrok URL and serving notebook.") from None


def ask_model(prompt, max_tokens=128, temperature=0):
    if not prompt.strip():
        raise ValueError("Enter a nonempty prompt.")
    started = time.perf_counter()
    response = send_request(
        "/v1/chat/completions",
        {
            "model": "nl2sh",
            "messages": [
                {"role": "system", "content": "Translate the request into a shell command. Return only the command."},
                {"role": "user", "content": prompt},
            ],
            "max_tokens": max_tokens,
            "temperature": temperature,
            "stream": False,
        },
    )
    reply = response["choices"][0]["message"]["content"]
    print("Prompt:", prompt)
    print("Reply:", reply)
    print(f"Time: {time.perf_counter() - started:.2f}s | Tokens: {response.get('usage', {})}")
    return response


In [ ]:
health = send_request("/health")
assert health.get("status") == "ok", health
print("Health check passed:")
print(json.dumps(health, indent=2))


## Enter your own prompt

Edit `PROMPT` below and run this cell as often as you like. `last_response`
contains the full API response; inspect it with
`print(json.dumps(last_response, indent=2))` if needed.

Use `temperature=0` for repeatable decoding and `max_tokens` between 1 and 512.
Input plus requested output must fit within the server's 4,096-token limit.


In [ ]:
PROMPT = "Find all Python files recursively in the current directory."

last_response = ask_model(PROMPT, max_tokens=128, temperature=0)


## Try several sample prompts

The next cell sends the requests one at a time, since the server handles one
generation at a time. Edit the list to compare responses for your own tasks.
Read the replies to judge whether they match the requests; a successful API
response alone does not prove that a generated command is correct.


In [ ]:
test_prompts = [
    "List all files including hidden files in the current directory.",
    "Find regular files larger than 100 MB recursively in the current directory.",
    "Count the number of lines in access.log that contain the word ERROR.",
]

test_responses = []
for prompt in test_prompts:
    test_responses.append(ask_model(prompt))
    print()
